# Week 1 · Day 2 — Lab 4
## Reductions, Aggregations, Sorting & Ranking

Once you can select data, the next move is to *collapse* it: a mean per column, a
max per row, the top-k responses, a sorted leaderboard. These are **reductions**
(many values → fewer) and **sorts/ranks** (reorder + locate). The two ideas that
trip people up — and that this lab drills — are the **`axis`** argument (which
direction does the array collapse?) and **missing data**: one stray `NaN`
silently poisons an ordinary `mean`/`sum`. You'll learn the NaN-safe family and
the index-returning functions (`argmax`, `argsort`) that power every ranking.

Your data is a 200×4 matrix of quality metrics for model responses, with some
readings missing.

### Learning objectives
1. Reduce along an **axis** and predict the output shape every time.
2. Recognize NaN poisoning and switch to the **NaN-safe** reductions (`nanmean`, `nansum`, `nanmax`).
3. Count missing values per column with `np.isnan(...).sum(axis=...)`.
4. Use `keepdims=True` so a reduction broadcasts cleanly back against the source.
5. Rank with `argmax`/`nanargmax`, `argsort`, and top-k slicing; carry a paired array along the sort.

### Time budget — ~70 min
| Segment | Time |
|---|---|
| Framing & objectives | 5 min |
| **A.** Axis-aware reductions | 12 min |
| **B.** NaN-safe stats & missing counts | 14 min |
| **C.** `keepdims` for broadcasting back | 10 min |
| **D.** argmax / argsort / top-k | 16 min |
| **E.** Sorting & paired ranking | 10 min |
| Wrap-up + stretch | 3 min |

### Files you need (in `data/`)
- `lab4_metric_matrix.npy` — shape (200, 4), `float64`, 4 quality metrics per response, some `NaN`.
- `lab4_records.npy` — structured array (`token_id` int32, `score` float32, `label` <U8), 12 rows (stretch only).


In [ ]:
import numpy as np
from pathlib import Path

print("NumPy", np.__version__)   # target curriculum: NumPy 2.x on Python 3.13

# Solution is different here because of folder structure

DATA = Path("../data")
if not DATA.exists():
    DATA = Path(".")

def check(label, predicate):
    try:
        ok = bool(predicate())
    except Exception as exc:
        ok = False
        label = f"{label}  (raised {type(exc).__name__}: {exc})"
    print(("PASS " if ok else "FAIL "), label)
    return ok

# 200 model responses x 4 quality metrics. Some cells are NaN (missing readings).
# Columns: [helpfulness, harmlessness, honesty, fluency], each roughly in [0, 1].
metrics = np.load(DATA / "lab4_metric_matrix.npy")
METRIC_NAMES = np.array(["helpfulness", "harmlessness", "honesty", "fluency"])

# A tiny structured array for the stretch section (mixed dtypes in one array).
records = np.load(DATA / "lab4_records.npy")   # fields: token_id, score, label

print("metrics:", metrics.shape, metrics.dtype)
print("missing cells (NaN):", int(np.isnan(metrics).sum()))
print("records:", records.shape, "fields:", records.dtype.names)

## Part A — Axis-aware reductions  *(guided)*

A reduction collapses one axis. The rule of thumb: **`axis` names the dimension
that disappears.** For a 2-D array shaped `(rows, cols)`:
- `axis=0` collapses *down the rows* → one value **per column** → shape `(cols,)`.
- `axis=1` collapses *across the columns* → one value **per row** → shape `(rows,)`.
- no axis → a single scalar over everything.


In [ ]:
small = np.array([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])
print("whole-array sum :", small.sum())            # 21.0  -> scalar
print("axis=0 (per col):", small.sum(axis=0))      # [5. 7. 9.] -> shape (3,)
print("axis=1 (per row):", small.sum(axis=1))      # [ 6. 15.]  -> shape (2,)

### Exercise A1 — Predict the shape, then a naive mean
First reduce the **clean** demo. Then try an ordinary `metrics.mean(axis=0)` on
the real (NaN-containing) matrix and observe what happens — set
`naive_col_means` and the boolean `naive_has_nan`.


In [ ]:
demo_col_sums = small.sum(axis=0)
demo_row_sums = small.sum(axis=1)

naive_col_means = metrics.mean(axis=0)
naive_has_nan = bool(np.isnan(naive_col_means).any())
print("demo_col_sums:", demo_col_sums, "| demo_row_sums:", demo_row_sums)
print("naive_col_means:", naive_col_means)
print("naive result poisoned by NaN?", naive_has_nan)

In [ ]:
check("A1: demo_col_sums == [5 7 9]", lambda: np.array_equal(demo_col_sums, [5, 7, 9]))
check("A1: demo_row_sums == [6 15]", lambda: np.array_equal(demo_row_sums, [6, 15]))
check("A1: naive mean WAS poisoned by NaN", lambda: naive_has_nan is True)

🧑‍🏫 **Instructor note — A1.** The payoff is `naive_has_nan is True`: an ordinary
`.mean(axis=0)` returns `NaN` for *every* column that has even one missing cell.
This is the motivating failure for Part B — don't fix it yet, let them feel it.
Reinforce the shape rule: `axis=0` → `(4,)` here, `axis=1` → `(200,)`.


## Part B — NaN-safe statistics & missing counts

Real eval data has gaps. The **NaN-safe** reductions — `np.nanmean`, `np.nansum`,
`np.nanmax`, `np.nanmin`, `np.nanstd` — ignore `NaN` instead of propagating it.
And `np.isnan(arr).sum(axis=...)` gives you a **missing-value count** per column
or row, which you should always log before trusting an average.


In [ ]:
col = np.array([0.8, np.nan, 0.6, 0.9])
print("plain mean :", col.mean())        # nan
print("nanmean    :", np.nanmean(col))   # 0.7666...
print("# missing  :", int(np.isnan(col).sum()))

### Exercise B1 — Trustworthy column statistics
Compute, ignoring missing cells:
- `col_means` — NaN-safe mean per metric (shape `(4,)`),
- `col_stds` — NaN-safe standard deviation per metric (shape `(4,)`),
- `missing_per_col` — count of missing cells per metric (shape `(4,)`, integer).


In [ ]:
col_means = np.nanmean(metrics, axis=0)
col_stds = np.nanstd(metrics, axis=0)
missing_per_col = np.isnan(metrics).sum(axis=0)
for name, m, s, miss in zip(METRIC_NAMES, col_means, col_stds, missing_per_col):
    print(f"{name:12s} mean={m:.3f}  std={s:.3f}  missing={int(miss)}")

In [ ]:
check("B1: col_means has shape (4,) and no NaN",
      lambda: col_means.shape == (4,) and not np.isnan(col_means).any())
check("B1: col_stds has shape (4,) and no NaN",
      lambda: col_stds.shape == (4,) and not np.isnan(col_stds).any())
check("B1: missing counts per col == [12, 10, 10, 12]",
      lambda: missing_per_col.tolist() == [12, 10, 10, 12])

### Exercise B2 — Per-response completeness
Going the other direction (`axis=1`), compute `missing_per_row` — how many of the
4 metrics are missing for each response — and `n_complete`, the number of
responses with **zero** missing metrics.


In [ ]:
missing_per_row = np.isnan(metrics).sum(axis=1)
n_complete = int((missing_per_row == 0).sum())
print("rows with 0/1/2 missing:",
      [int((missing_per_row == k).sum()) for k in (0, 1, 2)])
print("fully-complete responses:", n_complete)

In [ ]:
check("B2: missing_per_row has shape (200,)", lambda: missing_per_row.shape == (200,))
check("B2: total missing matches the matrix",
      lambda: int(missing_per_row.sum()) == int(np.isnan(metrics).sum()))
check("B2: n_complete counts zero-missing rows",
      lambda: n_complete == int((np.isnan(metrics).sum(axis=1) == 0).sum()))

🧑‍🏫 **Instructor note — B.** Expected missing-per-col is `[12, 10, 10, 12]`
(44 total). Stress the habit: *count missing before you average.* `nanmean` on a
column that is mostly missing returns a confident-looking number computed from
almost nothing — the count is your guardrail. There are no all-NaN rows or
columns in this dataset, so every NaN-safe reduction is well-defined (no
"Mean of empty slice" warning).


## Part C — `keepdims=True` for broadcasting back

When you reduce and then want to combine the result with the *original* array
(e.g. center each column by subtracting its mean), the reduced array has the
wrong number of dimensions to broadcast. `keepdims=True` keeps the collapsed axis
as a length-1 dimension so it lines up automatically.


In [ ]:
m = np.array([[1.0, 10.0],
              [3.0, 30.0]])
plain = m.mean(axis=0)                 # shape (2,)
kept  = m.mean(axis=0, keepdims=True)  # shape (1, 2)
print("plain shape:", plain.shape, "| keepdims shape:", kept.shape)
print("centered:\n", m - kept)        # broadcasts (2,2) - (1,2) cleanly

### Exercise C1 — Mean-center the columns
Using `np.nanmean(..., keepdims=True)`, build `col_means_kd` with shape `(1, 4)`,
then `centered = metrics - col_means_kd`. Verify each column of `centered` has a
NaN-safe mean of ~0.


In [ ]:
col_means_kd = np.nanmean(metrics, axis=0, keepdims=True)
centered = metrics - col_means_kd
check_means = np.nanmean(centered, axis=0)
print("col_means_kd shape:", col_means_kd.shape)
print("centered column means (~0):", np.round(check_means, 12))

In [ ]:
check("C1: col_means_kd shape is (1, 4)", lambda: col_means_kd.shape == (1, 4))
check("C1: centered keeps original shape", lambda: centered.shape == metrics.shape)
check("C1: centered column means are ~0",
      lambda: bool(np.allclose(check_means, 0.0, atol=1e-9)))

🧑‍🏫 **Instructor note — C1.** Without `keepdims`, `metrics - metrics.mean(axis=0)`
*happens* to work here only because the trailing dims align (200,4)-(4,) — but the
moment you reduce `axis=1` it breaks. Teach `keepdims=True` as the always-safe
habit so broadcasting is predictable regardless of axis. Centered means are ~1e-16,
i.e. floating-point zero.


## Part D — argmax / argsort / top-k

Often you want **where** the best value is, not the value itself. The `arg*`
family returns *indices*:
- `np.argmax` / `np.nanargmax` → index of the max (NaN-safe variant skips NaN),
- `np.argsort` → indices that would sort the array (ascending),
- top-k → `argsort(...)[-k:][::-1]` (last k, reversed to descending).

We rank on a per-response **composite** score = NaN-safe mean across the 4
metrics. (It has no NaN, because no response is missing all four.)


In [ ]:
v = np.array([0.3, 0.9, 0.1, 0.7])
print("argmax  :", np.argmax(v))           # 1
print("argsort :", np.argsort(v))          # [2 0 3 1]  (ascending)
print("top-2   :", np.argsort(v)[-2:][::-1])  # [1 3]  (descending)

### Exercise D1 — Composite score & the single best response
Build `composite` (shape `(200,)`) as the NaN-safe row mean of `metrics`. Then
find `best_idx`, the index of the highest-scoring response, and `best_score`.


In [ ]:
composite = np.nanmean(metrics, axis=1)
best_idx = int(np.argmax(composite))
best_score = float(composite[best_idx])
print(f"best response: index {best_idx}  composite {best_score:.4f}")

In [ ]:
check("D1: composite shape (200,) with no NaN",
      lambda: composite.shape == (200,) and not np.isnan(composite).any())
check("D1: best_idx is the argmax of composite",
      lambda: best_idx == int(np.argmax(composite)))
check("D1: best_score == composite[best_idx]",
      lambda: np.isclose(best_score, composite[best_idx]))

### Exercise D2 — Top-5 leaderboard
Produce `top5_idx`: the indices of the 5 highest composite scores, in
**descending** order. Then `top5_scores` is just `composite[top5_idx]`.


In [ ]:
top5_idx = np.argsort(composite)[-5:][::-1]
top5_scores = composite[top5_idx]
for rank, (i, s) in enumerate(zip(top5_idx, top5_scores), 1):
    print(f"#{rank}: response {i:3d}  composite {s:.4f}")

In [ ]:
check("D2: top5_idx has 5 entries", lambda: np.asarray(top5_idx).shape == (5,))
check("D2: top5_scores are sorted descending",
      lambda: bool(np.all(np.diff(top5_scores) <= 0)))
check("D2: #1 equals the global best",
      lambda: int(top5_idx[0]) == int(np.argmax(composite)))

### Exercise D3 — Best response per metric (NaN-safe)
For each of the 4 metrics, which response scored highest? Because columns contain
`NaN`, use **`np.nanargmax(..., axis=0)`** (plain `argmax` would point at a NaN).
Result `best_per_metric` has shape `(4,)`.


In [ ]:
best_per_metric = np.nanargmax(metrics, axis=0)
for name, idx in zip(METRIC_NAMES, best_per_metric):
    print(f"{name:12s}: response {int(idx):3d}  ({metrics[idx, list(METRIC_NAMES).index(name)]:.3f})")

In [ ]:
check("D3: best_per_metric shape (4,)", lambda: best_per_metric.shape == (4,))
check("D3: none of the picks is a NaN cell",
      lambda: not bool(np.isnan(metrics[best_per_metric, np.arange(4)]).any()))

🧑‍🏫 **Instructor note — D.** The headline trap is D3: plain `np.argmax` on a
column with NaN returns the index of a `NaN` (NaN compares as "not less than"),
silently giving a wrong winner. `np.nanargmax` is the fix. Tie this back to AI
evals: "top-k responses by score" is the literal operation behind reranking and
best-of-N sampling.


## Part E — Sorting & paired ranking

`np.sort` returns sorted **values**; `np.argsort` returns the **indices** that
sort the array — and those indices are how you reorder a *second*, parallel array
so it stays aligned. That "carry a companion array along the sort" move is the
core of building any leaderboard.


In [ ]:
ids = np.array(["a", "b", "c", "d"])
scores = np.array([0.3, 0.9, 0.1, 0.7])
order = np.argsort(scores)[::-1]          # descending
print("ranked ids   :", ids[order])       # ['b' 'd' 'a' 'c']
print("ranked scores:", scores[order])

### Exercise E1 — Build the leaderboard
Treat each response's position `0..199` as its id (`response_ids = np.arange(200)`).
Rank **all** responses by `composite` descending: produce `ranked_ids` and the
aligned `ranked_scores`. Confirm `ranked_ids[0]` is the same `best_idx` from D1.


In [ ]:
response_ids = np.arange(200)
order = np.argsort(composite)[::-1]
ranked_ids = response_ids[order]
ranked_scores = composite[order]
print("leaderboard top 3 :", ranked_ids[:3], ranked_scores[:3].round(4))
print("leaderboard bottom 3:", ranked_ids[-3:], ranked_scores[-3:].round(4))

In [ ]:
check("E1: ranked_scores descending",
      lambda: bool(np.all(np.diff(ranked_scores) <= 0)))
check("E1: ranked_ids is a permutation of 0..199",
      lambda: np.array_equal(np.sort(ranked_ids), np.arange(200)))
check("E1: top of leaderboard == D1 best_idx",
      lambda: int(ranked_ids[0]) == best_idx)

🧑‍🏫 **Instructor note — E1.** The pairing pattern `companion[np.argsort(key)]`
is worth saying out loud — students will reuse it constantly (sort filenames by
timestamp, reorder embeddings by relevance, align labels to predictions). Note
`np.argsort` is ascending-only; descending = reverse with `[::-1]` (or negate the
key). For stable ties, mention `kind="stable"` exists.


## Stretch goals *(for fast finishers)* — structured arrays

`lab4_records.npy` is a **structured array**: one array, multiple named
fields of different dtypes (`token_id` int32, `score` float32, `label` <U8).
You index fields by name: `records["score"]`.

**S1 — Sort records by score.** Use `np.argsort(records["score"])[::-1]` to get
`records_by_score`, the records ordered highest-score first.

**S2 — Filter by label.** Build `positives`, the subset of records whose `label`
is `"positive"`, and `top_positive`, the single positive record with the highest
score.


In [ ]:
order = np.argsort(records["score"])[::-1]
records_by_score = records[order]
print("S1 top record:", records_by_score[0])

positives = records[records["label"] == "positive"]
top_positive = positives[np.argmax(positives["score"])]
print("S2 #positives:", positives.shape[0], "| top positive:", top_positive)

In [ ]:
check("S1: records_by_score sorted by score desc",
      lambda: bool(np.all(np.diff(records_by_score["score"]) <= 0)))
check("S2: every record in positives is labelled positive",
      lambda: bool((positives["label"] == "positive").all()))
check("S2: top_positive is the max-score positive",
      lambda: float(top_positive["score"]) == float(positives["score"].max()))

🧑‍🏫 **Instructor note — Stretch.** Structured arrays are the bridge to pandas
(arriving on a later day): named columns, mixed dtypes, one container. Indexing a
field by name returns a normal 1-D array you can `argsort`/mask as usual. If time
is short, this section is safe to assign as take-home.


## Wrap-up — what you can now do

- Reduce along `axis=0` (per column) and `axis=1` (per row) and predict the output shape.
- Spot NaN poisoning and reach for `np.nanmean` / `np.nansum` / `np.nanmax` / `np.nanstd`.
- Count missing values per column and per row before trusting an average.
- Use `keepdims=True` so a reduction broadcasts back against the source array.
- Rank with `argmax` / `nanargmax`, `argsort`, top-k slicing, and carry a paired array along the sort.

**Next:** Lab 5 — reproducible randomness with the modern `default_rng`, then an
end-to-end eval-score capstone that ties the whole day together.
